# Оценка метрик поиска по JSONL-коллекции

Для каждого чанка используем `title` как запрос и проверяем, нашёлся ли сам чанк в top-K результатах.

**Метрики:** Recall@1, Recall@3, Recall@5, MRR

**ClearML:** результаты логируются в эксперимент для сравнения между прогонами.

In [5]:
# Настройки
JSONL_PATH   = "/workspace/data/processed/v2_podpunkti.jsonl"  # путь к файлу
COLLECTION   = "v2_podpunkti"                                   # коллекция Qdrant
EMBED_MODEL  = "bge-m3:latest"
OLLAMA_URL   = "http://ollama_cpu:11434"   # CPU-инстанс для эмбеддингов
LLM_URL      = "http://ollama:11434"       # GPU-инстанс для генерации
LLM_MODEL    = "bambucha/saiga-llama3:8b-q4_K"
QDRANT_URL   = "http://qdrant:6333"
TOP_K        = 5
SAMPLE       = 0   # 0 = все чанки, иначе случайная выборка N штук

# ClearML
CLEARML_PROJECT = "RAG-Evaluation"
CLEARML_TASK    = f"eval_{COLLECTION}"

In [6]:
import os
from clearml import Task

# Если нет API-ключей — работаем в offline-режиме (метрики в /tmp/clearml)
if not os.getenv("CLEARML_API_ACCESS_KEY"):
    Task.set_offline(True)
    print("ClearML: offline-режим (нет API-ключей)")
    print("Для облачного режима: зарегистрируйтесь на https://app.clear.ml")
    print("и задайте CLEARML_API_ACCESS_KEY / CLEARML_API_SECRET_KEY")

task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK,
    task_type=Task.TaskTypes.testing,
)

task.connect({
    "jsonl_path": JSONL_PATH,
    "collection": COLLECTION,
    "embed_model": EMBED_MODEL,
    "top_k": TOP_K,
    "sample": SAMPLE,
    "llm_model": LLM_MODEL,
})
logger = task.get_logger()
print(f"ClearML Task: {task.name} (id: {task.id})")

ModuleNotFoundError: No module named 'clearml'

In [ ]:
import json, random, time
import requests
import numpy as np
import pandas as pd
from IPython.display import display

# Загрузка файла
with open(JSONL_PATH, encoding="utf-8") as f:
    items = [json.loads(line) for line in f if line.strip()]

# Фильтруем: нужны чанки с title
items = [it for it in items if it.get("title", "").strip() and it.get("text", "").strip()]

if SAMPLE and SAMPLE < len(items):
    random.seed(42)
    items = random.sample(items, SAMPLE)

print(f"Чанков для оценки: {len(items)}")

In [ ]:
def embed(text: str) -> list:
    r = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": EMBED_MODEL, "input": text},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["embeddings"][0]

def search(vector: list, collection: str, top_k: int) -> list:
    r = requests.post(
        f"{QDRANT_URL}/collections/{collection}/points/search",
        json={"vector": vector, "limit": top_k, "with_payload": True},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["result"]

# Тест связи
try:
    emb = embed("тест")
    print(f"Ollama OK, dim={len(emb)}")
except Exception as e:
    print(f"Ошибка Ollama: {e}")

try:
    r = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}", timeout=5)
    cnt = r.json()["result"]["points_count"]
    print(f"Qdrant OK, коллекция '{COLLECTION}': {cnt} точек")
except Exception as e:
    print(f"Ошибка Qdrant: {e}")

In [ ]:
def recall_at_k(results, k):
    found = sum(1 for r in results if r["rank"] is not None and r["rank"] <= k)
    return found / len(results) if results else 0

def mrr(results):
    rr = [1 / r["rank"] if r["rank"] is not None else 0 for r in results]
    return np.mean(rr) if rr else 0

n = len(results)
metrics = {
    "Чанков оценено":  n,
    "Recall@1":        round(recall_at_k(results, 1), 4),
    "Recall@3":        round(recall_at_k(results, 3), 4),
    "Recall@5":        round(recall_at_k(results, 5), 4),
    "MRR":             round(mrr(results), 4),
    "Не найдено":      sum(1 for r in results if r["rank"] is None),
}

display(pd.DataFrame([metrics]).T.rename(columns={0: "Значение"}))

# --- ClearML: логируем метрики ---
for name in ("Recall@1", "Recall@3", "Recall@5", "MRR"):
    logger.report_scalar("Self-Retrieval", name, iteration=0, value=metrics[name])
logger.report_scalar("Self-Retrieval", "Не найдено", iteration=0, value=metrics["Не найдено"])

# Таблица результатов как артефакт
task.upload_artifact("self_retrieval_results", pd.DataFrame(results))

In [ ]:
def recall_at_k(results, k):
    found = sum(1 for r in results if r["rank"] is not None and r["rank"] <= k)
    return found / len(results) if results else 0

def mrr(results):
    rr = [1 / r["rank"] if r["rank"] is not None else 0 for r in results]
    return np.mean(rr) if rr else 0

n = len(results)
metrics = {
    "Чанков оценено":  n,
    "Recall@1":        round(recall_at_k(results, 1), 4),
    "Recall@3":        round(recall_at_k(results, 3), 4),
    "Recall@5":        round(recall_at_k(results, 5), 4),
    "MRR":             round(mrr(results), 4),
    "Не найдено":      sum(1 for r in results if r["rank"] is None),
}

display(pd.DataFrame([metrics]).T.rename(columns={0: "Значение"}))

In [ ]:
# Распределение рангов
import matplotlib.pyplot as plt

rank_counts = df["rank"].value_counts().sort_index()
labels = [str(r) if r <= TOP_K else "не найдено" for r in rank_counts.index]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, rank_counts.values, color="steelblue")
ax.set_xlabel("Rank")
ax.set_ylabel("Количество чанков")
ax.set_title(f"Распределение рангов — {COLLECTION}")
fig.tight_layout()

# ClearML: логируем график
logger.report_matplotlib_figure("Self-Retrieval", "Rank Distribution", figure=fig)
plt.show()

In [ ]:
# Распределение рангов
import matplotlib.pyplot as plt

rank_counts = df["rank"].value_counts().sort_index()
labels = [str(r) if r <= TOP_K else "не найдено" for r in rank_counts.index]

plt.figure(figsize=(8, 4))
plt.bar(labels, rank_counts.values, color="steelblue")
plt.xlabel("Rank")
plt.ylabel("Количество чанков")
plt.title(f"Распределение рангов — {COLLECTION}")
plt.tight_layout()
plt.show()

---
## Часть 2: Оценка по реальным вопросам

LLM генерирует вопрос по каждому чанку → поиск → проверяем нашёлся ли нужный чанк.  
Результат сохраняется в `qa_pairs.jsonl` — можно редактировать вручную.

In [ ]:
import os, pathlib

QA_PATH = os.path.join(os.path.dirname(JSONL_PATH), f"qa_pairs_{COLLECTION}.jsonl")

def generate_question(text: str) -> str:
    """LLM генерирует один вопрос по тексту чанка."""
    prompt = (
        "Прочитай текст и сформулируй один конкретный вопрос, "
        "на который этот текст отвечает. Выведи ТОЛЬКО вопрос, без пояснений.\n\n"
        f"Текст:\n{text[:2000]}"
    )
    r = requests.post(
        f"{LLM_URL}/api/generate",
        json={"model": LLM_MODEL, "prompt": prompt, "stream": False},
        timeout=180,
    )
    r.raise_for_status()
    return r.json()["response"].strip()

# Загружаем существующие пары, чтобы не генерировать повторно
existing_qa = {}
if os.path.exists(QA_PATH):
    with open(QA_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                existing_qa[rec["chunk_idx"]] = rec
    print(f"Загружено {len(existing_qa)} готовых QA-пар из {QA_PATH}")

qa_pairs = list(existing_qa.values())
new_count = 0

for i, item in enumerate(items):
    if i in existing_qa:
        continue
    try:
        question = generate_question(item["text"])
    except Exception as e:
        print(f"  [{i}] Ошибка генерации: {e}")
        continue

    pair = {
        "chunk_idx": i,
        "question": question,
        "true_id": str(item.get("id", "")),
        "chunk_text": item["text"][:500],
        "title": item.get("title", ""),
    }
    qa_pairs.append(pair)
    new_count += 1

    # Сохраняем инкрементально
    with open(QA_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

    if (new_count) % 10 == 0:
        print(f"  Сгенерировано: {new_count}, всего: {i+1}/{len(items)}")

print(f"\nНовых вопросов: {new_count}, всего QA-пар: {len(qa_pairs)}")
print(f"Файл: {QA_PATH}")

In [ ]:
# Оценка поиска по LLM-вопросам
qa_results = []
qa_errors = 0

for pair in qa_pairs:
    query = pair["question"]
    true_id = pair["true_id"]
    true_text = pair["chunk_text"]

    try:
        vec = embed(query)
        hits = search(vec, COLLECTION, TOP_K)
    except Exception as e:
        qa_errors += 1
        continue

    rank = None
    for k, h in enumerate(hits, start=1):
        payload = h.get("payload", {})
        hit_id = str(payload.get("id", ""))
        hit_text = payload.get("text", "")
        if (true_id and hit_id == true_id) or true_text in hit_text or hit_text in true_text:
            rank = k
            break

    qa_results.append({
        "query": query,
        "true_id": true_id,
        "title": pair.get("title", ""),
        "rank": rank,
        "score_1": hits[0]["score"] if hits else 0,
    })

print(f"Оценено: {len(qa_results)}, ошибок: {qa_errors}")

In [ ]:
# Метрики по LLM-вопросам
qa_metrics = {
    "Вопросов оценено": len(qa_results),
    "Recall@1":         round(recall_at_k(qa_results, 1), 4),
    "Recall@3":         round(recall_at_k(qa_results, 3), 4),
    "Recall@5":         round(recall_at_k(qa_results, 5), 4),
    "MRR":              round(mrr(qa_results), 4),
    "Не найдено":       sum(1 for r in qa_results if r["rank"] is None),
}

display(pd.DataFrame([qa_metrics]).T.rename(columns={0: "Значение"}))

# ClearML: логируем метрики LLM-вопросов
for name in ("Recall@1", "Recall@3", "Recall@5", "MRR"):
    logger.report_scalar("LLM-Questions", name, iteration=0, value=qa_metrics[name])
logger.report_scalar("LLM-Questions", "Не найдено", iteration=0, value=qa_metrics["Не найдено"])

task.upload_artifact("qa_results", pd.DataFrame(qa_results))
task.upload_artifact("qa_pairs_file", QA_PATH)

In [ ]:
# Сравнение: Self-Retrieval (title) vs LLM-Questions
comparison = pd.DataFrame({
    "Self-Retrieval (title)": {k: v for k, v in metrics.items() if k != "Чанков оценено"},
    "LLM-Questions":         {k: v for k, v in qa_metrics.items() if k != "Вопросов оценено"},
})

display(comparison)

# ClearML: таблица сравнения
logger.report_table("Comparison", "Self-Retrieval vs LLM-Questions", table_plot=comparison)

# Финализация
task.close()
print("ClearML эксперимент завершён.")